In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from category_encoders import OrdinalEncoder
from matplotlib.pylab import rand

честно не понял какой код надо было -дописать- с лекции (пытался найти, не нашел)

In [2]:
class MyGBRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.f0 = None

    def fit(self, X, y):
        self.f0 = np.mean(y)
        F_m = np.full_like(y, self.f0, dtype=np.float64)
        
        for _m in range(self.n_estimators):
            residuals = y - F_m
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F_m += self.learning_rate * tree.predict(X)

    def predict(self, X):
        predictions = np.full((X.shape[0],), self.f0, dtype=np.float64)
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)
        return predictions
    

In [3]:
df = pd.read_csv('ensembles-data-1.csv')

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42)

model = MyGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print('Basic GBRegressor Results:')
e = mean_squared_error(y_test, pred)
print(f'Mean Squared Error: {e}')
print(f'R^2 Score: {r2_score(y_test, pred)}')
print(f'Mean Absolute Error: {mean_absolute_error(y_test, pred)}')

Basic GBRegressor Results:
Mean Squared Error: 0.024846352758768397
R^2 Score: 0.9661186098744068
Mean Absolute Error: 0.08225402136636234


In [5]:
#с добавленными фичами
#subsample, colsample_bytree, feature_importances_, категориальные признаки


class MyGBRegressorAdvanced:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample = 1.0, colsample_bytree = 1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.trees = []
        self.tree_features = []
        self.f0 = None
        self.encoder = None
        self.cat_cols = []
        self.n_features_ = None

    #обработка категориальных признаков
    def _preprocess(self, X, is_fit=False):
        if isinstance(X, pd.DataFrame):
            X_copied = X.copy()
        else:
            X_copied = np.array(X, copy=True)

        if is_fit:
            if isinstance(X, np.ndarray):
                self.cat_cols = [i for i in range(X.shape[1]) if isinstance(X[0, i], str)]
            else:
                self.cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

        if len(self.cat_cols) > 0:
            self.encoder = OrdinalEncoder()
            if isinstance(X, np.ndarray):
                X_copied[:, self.cat_cols] = self.encoder.fit_transform(X[:, self.cat_cols])
            else:
                X_copied[self.cat_cols] = self.encoder.fit_transform(X[self.cat_cols])
        else:
            if self.encoder is not None and len(self.cat_cols) > 0:
                if isinstance(X, np.ndarray):
                    X_copied[:, self.cat_cols] = self.encoder.transform(X[:, self.cat_cols])
                else:
                    X_copied[self.cat_cols] = self.encoder.transform(X[self.cat_cols])

        if isinstance(X_copied, pd.DataFrame):
            return X_copied.to_numpy(dtype=np.float64)
        else:
            return np.asarray(X_copied, dtype=np.float64)
    
    
    def fit(self, X, y):
        X_processed = self._preprocess(X, is_fit=True)
        y_arr = np.asarray(y, dtype=np.float64)
        
        self.f0 = np.mean(y_arr)
        F_m = np.full_like(y_arr, self.f0, dtype=np.float64)
        n_samples, n_features = X_processed.shape
        self.n_features_ = n_features

        for _m in range(self.n_estimators):
            residuals = y_arr - F_m
            
            # Subsamples по строкам
            if self.subsample < 1.0:
                indices = np.random.choice(n_samples, size=int(n_samples * self.subsample), replace=False)
                X_b = X_processed[indices]
                res_b = residuals[indices]
            else:
                X_b = X_processed
                res_b = residuals

            # Subsamples по колонкам
            if self.colsample_bytree < 1.0:
                feature_indices = np.random.choice(n_features, size=int(n_features * self.colsample_bytree), replace=False)
            else:
                feature_indices = np.arange(n_features)

            X_b_sub = X_b[:, feature_indices]

            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42 + _m)
            tree.fit(X_b_sub, res_b)

            F_m += self.learning_rate * tree.predict(X_processed[:, feature_indices])

            self.trees.append(tree)
            self.tree_features.append(feature_indices)

    def predict(self, X):
        X_processed = self._preprocess(X, is_fit=False)
        predictions = np.full((X_processed.shape[0],), self.f0, dtype=np.float64)
        for tree, feature_indices in zip(self.trees, self.tree_features):
            predictions += self.learning_rate * tree.predict(X_processed[:, feature_indices])
        return predictions
    
    #feature_importances_
    @property
    def feature_importances_(self):
        if not self.trees:
            raise ValueError("The model has not been fitted yet.")
        
        importances = np.zeros(self.n_features_)

        for tree, feature_indices in zip(self.trees, self.tree_features):
            tree_importances = tree.feature_importances_
            for local_idx, global_idx in enumerate(feature_indices):
                importances[global_idx] += tree_importances[local_idx] * self.learning_rate

        #нормализация
        sum_importances = np.sum(importances)
        if (sum_importances > 0):
            importances /= sum_importances

        return importances
    

In [6]:
df = pd.read_csv('ensembles-data-1.csv')
#для начала проверим на том же датасете что и первая (для сравнения)
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42)

model = MyGBRegressorAdvanced(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Advanced GBRegressor results (basic):")
e = mean_squared_error(y_test, pred)
print(f'Mean Squared Error: {e}')
print(f'R^2 Score: {r2_score(y_test, pred)}')
print(f'Mean Absolute Error: {mean_absolute_error(y_test, pred)}\n\n')

model = MyGBRegressorAdvanced(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, colsample_bytree=0.8)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Advanced GBRegressor results (with subsample and colsample_bytree):")
e = mean_squared_error(y_test, pred)
print(f'Mean Squared Error: {e}')
print(f'R^2 Score: {r2_score(y_test, pred)}')
print(f'Mean Absolute Error: {mean_absolute_error(y_test, pred)}')

#объяснение - 0.8 для subsample и colsample_bytree означает, что на каждом шаге обучения будет использоваться 80% случайно выбранных строк и 80% случайно выбранных признаков для построения дерева.
#это неприемлемо для нашего датасета из 2х признаков, так как при таком подходе может быть выбрано только 1 признак, что приводит к потере информации и ухудшению качества модели:(


Advanced GBRegressor results (basic):
Mean Squared Error: 0.024906059660980535
R^2 Score: 0.9660371913713902
Mean Absolute Error: 0.08269305522302696


Advanced GBRegressor results (with subsample and colsample_bytree):
Mean Squared Error: 0.10192241736477228
R^2 Score: 0.8610148854116741
Mean Absolute Error: 0.18170512159365465


In [7]:
#для полной проверки берем датасет с категориальными колонками - toyota corolla

df = pd.read_csv('ToyotaCorolla.csv')

X = df.drop(columns=['Price', 'Id'], errors='ignore')
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MyGBRegressorAdvanced(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.7
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score: {r2:.4f}")
print(f"MAE:      {mae:.2f}")
print(f"RMSE:     {rmse:.2f}")


df_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nТоп-5 самых важных признаков")
print(df_imp.head(5).to_string(index=False))

R² Score: 0.9388
MAE:      695.99
RMSE:     903.54

Топ-5 самых важных признаков
  Feature  Importance
Age_08_04    0.199360
       KM    0.155149
   Weight    0.110001
 Mfg_Year    0.083666
    Model    0.075707
